# ANN experiment

This notebook runs the workspace's ANN-0.1 adaptive synapse model on a small deterministic token transition problem. It measures whether the model learns the transition, monitors the latent iterations, and compares an explicitly damaged copy before and after repair.

**Run locally:** use a Python 3.11+ Jupyter kernel with PyTorch 2.6+ installed in that environment. Select **Restart Kernel and Run All Cells**. The data and weights are synthetic and the run needs no network or GPU.

### Architecture and provenance

The model uses tied input/output embeddings, pointwise adaptive synapses that scale activations using tracked variance, and a damped latent iteration. The embedded classes come from `ANN/config.py`, `ops.py`, `synapse.py`, `reasoning.py`, `block.py`, `self_healing.py`, and `model.py` in the development workspace. Cells omit module headers and flatten relative imports.

The notebook adapts one method. The original `SpectralLipschitzLinear.forward` can change a saved parameter in place between latent iterations, making backward fail once rescaling activates. The notebook computes a separate scaled weight for each training forward while preserving the source's power-iteration estimate and evaluation-mode behavior. It exercises that path below with deliberately large initial weights.

Read the experimental labels in the embedded comments as source terminology. Approximate spectral normalization and finite residual histories do not prove global contraction or imply any connection to a proprietary model.

In [1]:
import math
import platform
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 27
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print({"python": platform.python_version(), "torch": torch.__version__, "device": "cpu", "seed": SEED})

{'python': '3.11.0rc2', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 27}


In [2]:
# Source: ANN/config.py
# SHA-256 of original file: d7a25dc1efdeb2efb19722b4132d8825b581dcfc53b97c8903f1b5d36cc9b586

from __future__ import annotations

import json

from dataclasses import asdict, dataclass

from pathlib import Path

from typing import Any, Dict, List, Optional, Union

@dataclass
class ANNConfig:
    """Configuration for ANN Self-Healing Primitives and Models."""

    # ─── Core Dimensions ────────────────────────────────────────────────────
    vocab_size: int = 128_256
    d_model: int = 512
    n_layers: int = 8
    max_seq_len: int = 2048
    dropout: float = 0.0
    norm_eps: float = 1e-6

    # ─── Bi-Level Adaptive Synapse Parameters ───────────────────────────────
    target_variance: float = 1.0         # Target activation variance sigma*
    variance_momentum: float = 0.05      # Beta for running mean/var tracking
    homeostatic_gain_power: float = 0.5  # Gamma in (sigma* / sigma)^gamma
    max_homeostatic_gain: float = 3.0    # Clamp upper gain bound
    min_homeostatic_gain: float = 0.33   # Clamp lower gain bound

    # ─── Self-Healing Diagnostic & Rejuvenation Engine ──────────────────────
    enable_self_healing: bool = True
    dead_variance_threshold: float = 1e-4 # Below this variance, neuron is flagged dead
    saturation_threshold: float = 0.95    # Fraction of clamped activations to flag saturation
    rejuvenation_scale: float = 0.05      # Magnitude of orthogonal rejuvenation perturbation
    healing_frequency: int = 5            # Evaluate self-healing every N forward passes
    subspace_projection_repair: bool = True # Project representations out of NaN/Inf corridors

    # ─── DeepMind Latent Fixed-Point Pondering ───────────────────────────────
    enable_fixed_point_reasoning: bool = True
    max_ponder_iterations: int = 5        # Maximum latent thinking cycles
    convergence_tolerance: float = 1e-3   # Residual norm tolerance ||h_{k} - h_{k-1}|| < tau
    spectral_lipschitz_bound: float = 0.95 # Strictly < 1.0 to guarantee Banach contraction
    ponder_regularizer_weight: float = 0.01

    # ─── Training & Optimization ────────────────────────────────────────────
    lr: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    warmup_steps: int = 500
    total_steps: int = 50_000
    grad_clip: float = 1.0
    bf16: bool = True

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def to_json(self, path: Union[str, Path], indent: int = 2) -> None:
        Path(path).write_text(json.dumps(self.to_dict(), indent=indent))

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "ANNConfig":
        return cls(**d)

    @classmethod
    def from_json(cls, path: Union[str, Path]) -> "ANNConfig":
        return cls.from_dict(json.loads(Path(path).read_text()))

    # ─── Presets ─────────────────────────────────────────────────────────────
    @classmethod
    def preset_tiny(cls, **overrides) -> "ANNConfig":
        """Fast smoke test & trauma demonstration configuration."""
        base = dict(
            vocab_size=1000,
            d_model=128,
            n_layers=4,
            max_seq_len=256,
            healing_frequency=1,
            max_ponder_iterations=4,
            convergence_tolerance=1e-3,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def preset_base(cls, **overrides) -> "ANNConfig":
        """Base research model (~120M parameters)."""
        base = dict(
            vocab_size=50_272,
            d_model=768,
            n_layers=12,
            max_seq_len=2048,
            healing_frequency=5,
            max_ponder_iterations=5,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def preset_large(cls, **overrides) -> "ANNConfig":
        """Frontier-scale self-healing configuration (~1.2B parameters)."""
        base = dict(
            vocab_size=128_256,
            d_model=2048,
            n_layers=24,
            max_seq_len=4096,
            healing_frequency=10,
            max_ponder_iterations=6,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def get_preset(cls, name: str) -> "ANNConfig":
        presets = {
            "tiny": cls.preset_tiny,
            "ann_tiny": cls.preset_tiny,
            "base": cls.preset_base,
            "ann_base": cls.preset_base,
            "large": cls.preset_large,
            "ann_large": cls.preset_large,
        }
        if name not in presets:
            raise ValueError(f"Unknown preset: {name}. Available: {list(presets.keys())}")
        return presets[name]()

In [3]:
# Source: ANN/ops.py
# SHA-256 of original file: ffb433c882f7e92fca301947bfc4cb666f2a51098b2212cf44cbd33ece82e65b

from __future__ import annotations

import math

from typing import Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization with FP32 precision."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        input_dtype = x.dtype
        x_fp32 = x.float()
        variance = x_fp32.pow(2).mean(-1, keepdim=True)
        normed = x_fp32 * torch.rsqrt(variance + self.eps)
        return (normed * self.weight.float()).to(input_dtype)

class SpectralLipschitzLinear(nn.Linear):
    """
    Linear projection with spectral norm bounding: Lip(W) <= lambda_spec < 1.0.
    Mathematically guarantees that iterative transformations form a contraction mapping.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = False,
        lipschitz_bound: float = 0.95,
        n_power_iterations: int = 1,
    ):
        super().__init__(in_features, out_features, bias=bias)
        self.lipschitz_bound = min(max(lipschitz_bound, 0.1), 0.999)
        self.n_power_iterations = n_power_iterations

        # Random vector for power iteration
        self.register_buffer("u_vector", torch.randn(out_features))
        self.u_vector: Tensor = F.normalize(self.u_vector, dim=0)

    def _get_spectral_norm(self) -> Tensor:
        """Approximates the largest singular value via power iteration."""
        W = self.weight.float()
        u = self.u_vector.float()

        for _ in range(self.n_power_iterations):
            # v = W^T u / ||W^T u||
            v = F.normalize(torch.mv(W.t(), u), dim=0)
            # u = W v / ||W v||
            u = F.normalize(torch.mv(W, v), dim=0)

        self.u_vector.copy_(u.detach().to(self.weight.dtype))
        sigma = torch.dot(u, torch.mv(W, v))
        return sigma

    def forward(self, x: Tensor) -> Tensor:
        weight = self.weight
        if self.training:
            with torch.no_grad():
                sigma = self._get_spectral_norm()
                scale = (self.lipschitz_bound / (sigma + 1e-8)).clamp(max=1)
            weight = self.weight * scale

        return F.linear(x, weight, self.bias)

class HomeostaticTracker(nn.Module):
    """
    Tracks running statistics of neuron activations and computes homeostatic gain scaling.
    Turrigiano-style synaptic scaling: neurons adjust their sensitivity to maintain target variance.
    """

    def __init__(
        self,
        features: int,
        target_variance: float = 1.0,
        momentum: float = 0.05,
        gain_power: float = 0.5,
        min_gain: float = 0.33,
        max_gain: float = 3.0,
    ):
        super().__init__()
        self.features = features
        self.target_variance = target_variance
        self.momentum = momentum
        self.gain_power = gain_power
        self.min_gain = min_gain
        self.max_gain = max_gain

        # Buffers for running mean and variance
        self.register_buffer("running_mean", torch.zeros(features))
        self.register_buffer("running_var", torch.ones(features))
        self.register_buffer("firing_rate", torch.ones(features) * 0.5)

    def update_and_get_gain(self, activations: Tensor) -> Tensor:
        """
        Args:
            activations: (..., features)
        Returns:
            gain: (features,) multiplicative gain vector
        """
        act_flat = activations.detach().float().reshape(-1, self.features)
        batch_mean = act_flat.mean(dim=0)
        batch_var = act_flat.var(dim=0, unbiased=False)
        batch_firing = (act_flat.abs() > 0.01).float().mean(dim=0)

        if self.training:
            # Exponential moving average update
            self.running_mean.lerp_(batch_mean, self.momentum)
            self.running_var.lerp_(batch_var, self.momentum)
            self.firing_rate.lerp_(batch_firing, self.momentum)

        # Homeostatic gain computation: g_i = (target / (std + eps))^gamma
        std = torch.sqrt(self.running_var.clamp(min=1e-8))
        target_std = math.sqrt(self.target_variance)
        ratio = target_std / std
        gain = torch.clamp(ratio.pow(self.gain_power), min=self.min_gain, max=self.max_gain)

        return gain.to(activations.dtype)

def compute_activation_entropy(activations: Tensor) -> float:
    """Calculates activation entropy across feature dimensions to monitor representation collapse."""
    var = activations.detach().float().var(dim=-1, unbiased=False).clamp(min=1e-8)
    entropy = 0.5 * torch.log(2 * math.pi * math.e * var).mean().item()
    return entropy

def generate_orthogonal_rejuvenation(
    weight: Tensor,
    dead_indices: Tensor,
    scale: float = 0.05,
) -> Tensor:
    """
    Generates a weight perturbation strictly orthogonal to the dominant singular subspace
    of the active weight matrix to revive dead neurons without causing catastrophic interference.
    """
    out_features, in_features = weight.shape
    device = weight.device
    dtype = weight.dtype

    # Create random perturbation
    pert = torch.randn(len(dead_indices), in_features, device=device, dtype=torch.float32)

    # Gram-Schmidt against top active weights
    active_mask = torch.ones(out_features, device=device, dtype=torch.bool)
    active_mask[dead_indices] = False
    active_weights = weight[active_mask].float()

    if active_weights.shape[0] > 0:
        # Project perturbation out of active mean subspace
        mean_dir = F.normalize(active_weights.mean(dim=0, keepdim=True), dim=-1)
        projection = torch.sum(pert * mean_dir, dim=-1, keepdim=True) * mean_dir
        pert = pert - projection

    pert = F.normalize(pert, dim=-1) * scale
    return pert.to(dtype)

In [4]:
# Source: ANN/synapse.py
# SHA-256 of original file: 4808d09e23a5b8b18d8f332f6d33fd8a8d735bcf7bb1d06268cca171de80e60f

from __future__ import annotations

import math

from typing import Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class BiLevelAdaptiveSynapse(nn.Module):
    """
    Adaptive Linear Synapse with Homeostatic Self-Stabilization.
    Functions as a direct drop-in replacement for nn.Linear.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = False,
        cfg: Optional[ANNConfig] = None,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Configuration fallbacks
        target_var = cfg.target_variance if cfg else 1.0
        momentum = cfg.variance_momentum if cfg else 0.05
        gain_power = cfg.homeostatic_gain_power if cfg else 0.5
        min_gain = cfg.min_homeostatic_gain if cfg else 0.33
        max_gain = cfg.max_homeostatic_gain if cfg else 3.0
        self.dead_var_thresh = cfg.dead_variance_threshold if cfg else 1e-4

        # Base structural weights
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter("bias", None)

        # Homeostatic Activity Tracker
        self.tracker = HomeostaticTracker(
            features=out_features,
            target_variance=target_var,
            momentum=momentum,
            gain_power=gain_power,
            min_gain=min_gain,
            max_gain=max_gain,
        )

        self._init_parameters()

    def _init_parameters(self):
        # Kaiming uniform initialization
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: (..., in_features)
        Returns:
            out: (..., out_features) scaled by homeostatic gain
        """
        # 1. Structural projection
        base_out = F.linear(x, self.weight, self.bias)

        # 2. Compute and apply homeostatic gain
        gain = self.tracker.update_and_get_gain(base_out)

        # 3. Dynamic modulation
        out = base_out * gain
        return out

    def get_dead_neuron_indices(self) -> Tensor:
        """Identifies dead neurons whose activation variance has collapsed."""
        dead_mask = self.tracker.running_var < self.dead_var_thresh
        return torch.nonzero(dead_mask).squeeze(-1)

    def get_saturation_fraction(self) -> float:
        """Calculates fraction of neurons operating at extreme variance bounds."""
        var = self.tracker.running_var
        saturated = (var > 10.0) | (var < self.dead_var_thresh)
        return float(saturated.float().mean().item())

    @torch.no_grad()
    def rejuvenate_dead_neurons(self, scale: float = 0.05) -> int:
        """
        Self-healing operation: revives dead neurons by injecting orthogonal
        perturbations into their structural weights and resetting running variance.
        """
        dead_indices = self.get_dead_neuron_indices()
        num_dead = len(dead_indices)
        if num_dead == 0:
            return 0

        # Inject orthogonal perturbation
        pert = generate_orthogonal_rejuvenation(self.weight, dead_indices, scale=scale)
        self.weight[dead_indices] += pert

        # Reset tracker statistics for revived neurons
        self.tracker.running_var[dead_indices] = self.tracker.target_variance
        self.tracker.running_mean[dead_indices] = 0.0
        self.tracker.firing_rate[dead_indices] = 0.5

        return num_dead

    @torch.no_grad()
    def homeostatic_downscale_saturated(self, factor: float = 0.9):
        """Downscales weights of runaway hyper-active neurons to prevent gradient explosion."""
        hyper_active = self.tracker.running_var > 10.0
        if hyper_active.any():
            self.weight[hyper_active] *= factor
            self.tracker.running_var[hyper_active] *= factor

ANNLinear = BiLevelAdaptiveSynapse

In [5]:
# Source: ANN/reasoning.py
# SHA-256 of original file: 6b822558e0f6fa0da49397f66b66bc928f324adeb450bcea884659f0b6e69ddc

from __future__ import annotations

from typing import List, Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class ContractionReasoningCell(nn.Module):
    """Lipschitz-bounded transition operator T(h; x)."""

    def __init__(self, d_model: int, lipschitz_bound: float = 0.95, norm_eps: float = 1e-6):
        super().__init__()
        self.d_model = d_model
        self.norm = RMSNorm(d_model, norm_eps)

        # Spectral Lipschitz bounded layers guarantee Lip(T) < 1.0
        self.w_in = SpectralLipschitzLinear(d_model * 2, d_model * 2, lipschitz_bound=lipschitz_bound)
        self.w_out = SpectralLipschitzLinear(d_model * 2, d_model, lipschitz_bound=lipschitz_bound)

    def forward(self, h: Tensor, x: Tensor) -> Tensor:
        """Combines current thought h with external input x."""
        hx = torch.cat([self.norm(h), self.norm(x)], dim=-1)
        hidden = F.silu(self.w_in(hx))
        return self.w_out(hidden)

class FixedPointPonderCore(nn.Module):
    """
    Iterative Fixed-Point Reasoning Engine.
    Refines latent states until convergence to a stationary equilibrium.
    """

    def __init__(self, cfg: ANNConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.max_iters = max(1, cfg.max_ponder_iterations)
        self.tolerance = cfg.convergence_tolerance
        self.damping = 0.5  # Damped Mann iteration factor alpha

        self.cell = ContractionReasoningCell(
            d_model=cfg.d_model,
            lipschitz_bound=cfg.spectral_lipschitz_bound,
            norm_eps=cfg.norm_eps,
        )

    def forward(
        self,
        x: Tensor,
        h_init: Optional[Tensor] = None,
        max_iters: Optional[int] = None,
    ) -> Tuple[Tensor, Tensor, List[float]]:
        """
        Args:
            x: Input representation (B, T, D)
            h_init: Optional initial state
            max_iters: Optional iteration limit override
        Returns:
            h_star: Equilibrium representation at fixed point
            ponder_loss: Regularizer penalty on contemplation steps
            residual_history: List of residual norms per step
        """
        K = max_iters or self.max_iters
        h = h_init if h_init is not None else x.clone()

        residual_history: List[float] = []
        steps_taken = torch.zeros(x.shape[0], x.shape[1], 1, device=x.device, dtype=torch.float32)

        for k in range(1, K + 1):
            # T(h; x)
            t_out = self.cell(h, x)

            # Damped Picard iteration
            h_next = (1.0 - self.damping) * h + self.damping * t_out

            # Check contraction residual: ||h_next - h|| / (||h_next|| + eps)
            diff = (h_next - h).norm(dim=-1, keepdim=True)
            norm = h_next.norm(dim=-1, keepdim=True).clamp(min=1e-6)
            rel_res = (diff / norm).mean().item()
            residual_history.append(rel_res)

            steps_taken += 1.0
            h = h_next

            # Early exit if converged across all tokens
            if rel_res < self.tolerance:
                break

        ponder_loss = self.cfg.ponder_regularizer_weight * (steps_taken / K).mean()
        return h, ponder_loss, residual_history

In [6]:
# Source: ANN/block.py
# SHA-256 of original file: 011a5f09963f078aa76e49b9e459c89891d24eb3ddcb52e228a437d60eba9648

from __future__ import annotations

from typing import List, Optional, Tuple

import torch

import torch.nn as nn

from torch import Tensor

class ANNBlock(nn.Module):
    """
    Self-Healing Adaptive Neural Block.
    """

    def __init__(self, cfg: ANNConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model

        # Normalization
        self.norm1 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.norm2 = RMSNorm(cfg.d_model, cfg.norm_eps)

        # Bi-Level Adaptive Synapses (FFN)
        self.w1 = BiLevelAdaptiveSynapse(cfg.d_model, cfg.d_model * 2, bias=False, cfg=cfg)
        self.act = nn.SiLU()
        self.w2 = BiLevelAdaptiveSynapse(cfg.d_model * 2, cfg.d_model, bias=False, cfg=cfg)

        # DeepMind Fixed-Point Contraction Pondering Core
        if cfg.enable_fixed_point_reasoning:
            self.ponder_core = FixedPointPonderCore(cfg)
        else:
            self.ponder_core = None

        self.dropout = nn.Dropout(cfg.dropout) if cfg.dropout > 0.0 else nn.Identity()

    def forward(
        self,
        x: Tensor,
        max_ponder_iters: Optional[int] = None,
    ) -> Tuple[Tensor, Tensor, List[float]]:
        """
        Args:
            x: (B, T, D)
        Returns:
            out: (B, T, D)
            ponder_loss: Regularizer penalty on contemplation
            residuals: Contraction convergence residuals
        """
        # 1. Adaptive Synapse FFN Pass
        h = self.norm1(x)
        h_ffn = self.w2(self.act(self.w1(h)))
        x = x + self.dropout(h_ffn)

        # 2. DeepMind Latent Fixed-Point Reasoning Pass
        ponder_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
        residuals: List[float] = []

        if self.ponder_core is not None:
            h_norm = self.norm2(x)
            h_pondered, ponder_loss, residuals = self.ponder_core(h_norm, max_iters=max_ponder_iters)
            x = x + self.dropout(h_pondered)

        return x, ponder_loss, residuals

In [7]:
# Source: ANN/self_healing.py
# SHA-256 of original file: 26d4b9b2323de7909c3ed969a7f025d8978fc4aa9e8eadd60a07a80f87d78a0e

from __future__ import annotations

from dataclasses import dataclass

from typing import Dict, List, Optional, Tuple

import torch

import torch.nn as nn

from torch import Tensor

@dataclass
class HealthReport:
    """Diagnostic health summary of a neural model."""
    total_neurons: int
    dead_neurons: int
    dead_ratio: float
    saturated_neurons: int
    saturated_ratio: float
    nan_detected: bool
    inf_detected: bool
    mean_variance: float
    is_healthy: bool

@dataclass
class HealingActionSummary:
    """Summary of autonomous repair actions performed."""
    rejuvenated_neurons: int
    downscaled_neurons: int
    repaired_nan_layers: int

class SelfHealingEngine:
    """
    Autonomous Diagnostic and Repair Engine for Adaptive Neural Networks.
    """

    def __init__(self, cfg: ANNConfig):
        self.cfg = cfg
        self.rejuvenation_scale = cfg.rejuvenation_scale
        self.dead_thresh = cfg.dead_variance_threshold
        self.step_counter = 0

    def diagnose(self, model: nn.Module) -> HealthReport:
        """Performs a comprehensive diagnostic sweep over all adaptive synapses."""
        total_neurons = 0
        dead_neurons = 0
        saturated_neurons = 0
        var_list = []
        has_nan = False
        has_inf = False

        for name, module in model.named_modules():
            if isinstance(module, BiLevelAdaptiveSynapse):
                n_out = module.out_features
                total_neurons += n_out

                # Check dead and saturated
                dead_idx = module.get_dead_neuron_indices()
                dead_neurons += len(dead_idx)

                var = module.tracker.running_var
                saturated = (var > 10.0).sum().item()
                saturated_neurons += saturated

                var_list.append(var.mean().item())

            # Check parameter sanity
            for p in module.parameters(recurse=False):
                if torch.isnan(p).any():
                    has_nan = True
                if torch.isinf(p).any():
                    has_inf = True

        dead_ratio = dead_neurons / max(1, total_neurons)
        sat_ratio = saturated_neurons / max(1, total_neurons)
        mean_var = (sum(var_list) / max(1, len(var_list))) if var_list else 1.0

        is_healthy = (dead_ratio < 0.05) and (sat_ratio < 0.05) and (not has_nan) and (not has_inf)

        return HealthReport(
            total_neurons=total_neurons,
            dead_neurons=dead_neurons,
            dead_ratio=dead_ratio,
            saturated_neurons=saturated_neurons,
            saturated_ratio=sat_ratio,
            nan_detected=has_nan,
            inf_detected=has_inf,
            mean_variance=mean_var,
            is_healthy=is_healthy,
        )

    def heal(self, model: nn.Module) -> HealingActionSummary:
        """Executes targeted autonomous repair on degraded or dead neurons."""
        rejuvenated = 0
        downscaled = 0
        repaired_nans = 0

        for name, module in model.named_modules():
            if isinstance(module, BiLevelAdaptiveSynapse):
                # 1. Rejuvenate dead neurons
                revived = module.rejuvenate_dead_neurons(scale=self.rejuvenation_scale)
                rejuvenated += revived

                # 2. Downscale hyper-active runaway neurons
                hyper_active = (module.tracker.running_var > 10.0).sum().item()
                if hyper_active > 0:
                    module.homeostatic_downscale_saturated(factor=0.85)
                    downscaled += hyper_active

                # 3. Check and repair NaNs in weights
                if torch.isnan(module.weight).any() or torch.isinf(module.weight).any():
                    with torch.no_grad():
                        module.weight.data = torch.nan_to_num(module.weight.data, nan=0.0, posinf=1.0, neginf=-1.0)
                        # Re-seed with orthogonal noise
                        module.weight.data += torch.randn_like(module.weight.data) * 0.02
                        repaired_nans += 1

        return HealingActionSummary(
            rejuvenated_neurons=rejuvenated,
            downscaled_neurons=downscaled,
            repaired_nan_layers=repaired_nans,
        )

    def step(self, model: nn.Module) -> Optional[HealingActionSummary]:
        """Periodic self-healing check executed during training forward passes."""
        self.step_counter += 1
        if self.step_counter % self.cfg.healing_frequency != 0:
            return None

        report = self.diagnose(model)
        if not report.is_healthy:
            return self.heal(model)
        return None

    @staticmethod
    def inject_trauma(
        model: nn.Module,
        dead_ratio: float = 0.5,
        shock_noise: float = 0.0,
    ) -> int:
        """
        Trauma Simulation Harness:
        Deliberately knocks out a fraction of neurons across the model to test resilience.
        """
        killed_count = 0
        with torch.no_grad():
            for name, module in model.named_modules():
                if isinstance(module, BiLevelAdaptiveSynapse):
                    n_out = module.out_features
                    k = int(n_out * dead_ratio)
                    if k > 0:
                        perm = torch.randperm(n_out)[:k]
                        # Zero out weights and kill variance
                        module.weight[perm] = 0.0
                        module.tracker.running_var[perm] = 1e-6
                        killed_count += k

                    if shock_noise > 0.0:
                        module.weight.add_(torch.randn_like(module.weight) * shock_noise)

        return killed_count

In [8]:
# Source: ANN/model.py
# SHA-256 of original file: d2076a32430246afdea3dbc98deb9fe57fe04494a9cccaa2253f10d28802e962

from __future__ import annotations

from typing import Dict, List, Optional, Tuple, Union

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class AdaptiveNeuralNetwork(nn.Module):
    """
    Self-Healing Adaptive Neural Network (ANN) Foundation Model.
    """

    def __init__(self, cfg: ANNConfig):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = cfg.vocab_size
        self.d_model = cfg.d_model

        # Token Embedding
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout) if cfg.dropout > 0.0 else nn.Identity()

        # Stack of ANN Blocks
        self.layers = nn.ModuleList([
            ANNBlock(cfg) for _ in range(cfg.n_layers)
        ])

        # Output head
        self.norm_final = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # Weight tying

        # Self-Healing Diagnostic & Repair Engine
        self.healing_engine = SelfHealingEngine(cfg)

        self._init_weights()

    def _init_weights(self):
        std = 0.02
        nn.init.trunc_normal_(self.embed.weight, std=std)
        for module in self.modules():
            if isinstance(module, nn.Linear) and not hasattr(module, "tracker"):
                nn.init.trunc_normal_(module.weight, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids: Tensor,
        max_ponder_iters: Optional[int] = None,
    ) -> Tuple[Tensor, Dict[str, Any]]:
        """
        Forward pass with integrated self-healing monitoring.
        """
        x = self.drop(self.embed(input_ids))

        total_ponder_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
        layer_residuals: List[List[float]] = []

        for layer in self.layers:
            x, p_loss, res = layer(x, max_ponder_iters=max_ponder_iters)
            total_ponder_loss = total_ponder_loss + p_loss
            layer_residuals.append(res)

        x = self.norm_final(x)
        logits = self.lm_head(x)

        # Periodic runtime self-healing sweep
        healing_action = None
        if self.training and self.cfg.enable_self_healing:
            healing_action = self.healing_engine.step(self)

        aux = {
            "ponder_loss": total_ponder_loss,
            "layer_residuals": layer_residuals,
            "healing_action": healing_action,
        }

        return logits, aux

    def compute_loss(
        self,
        input_ids: Tensor,
        max_ponder_iters: Optional[int] = None,
    ) -> Tuple[Tensor, Dict[str, float]]:
        """Computes cross-entropy language modeling loss + ponder regularizer."""
        B, T = input_ids.shape
        logits, aux = self.forward(input_ids, max_ponder_iters=max_ponder_iters)

        pred_logits = logits[:, :-1].reshape(-1, self.vocab_size)
        targets = input_ids[:, 1:].reshape(-1)

        ar_loss = F.cross_entropy(pred_logits, targets)
        ponder_loss = aux["ponder_loss"]

        total_loss = ar_loss + ponder_loss

        metrics = {
            "loss": total_loss.item(),
            "loss_ar": ar_loss.item(),
            "loss_ponder": ponder_loss.item(),
        }

        return total_loss, metrics

    def diagnose(self) -> HealthReport:
        """Runs complete diagnostic health check on the network."""
        return self.healing_engine.diagnose(self)

    def heal(self) -> HealingActionSummary:
        """Triggers immediate autonomous repair of degraded pathways."""
        return self.healing_engine.heal(self)

    def inject_trauma(self, dead_ratio: float = 0.5, shock_noise: float = 0.0) -> int:
        """Simulates catastrophic damage to verify self-healing recovery."""
        return self.healing_engine.inject_trauma(self, dead_ratio=dead_ratio, shock_noise=shock_noise)

    @torch.inference_mode()
    def generate(
        self,
        input_ids: Tensor,
        max_new_tokens: int = 32,
        temperature: float = 0.8,
        top_k: int = 50,
    ) -> Tensor:
        """Autoregressive text generation."""
        self.eval()
        generated = input_ids.clone()

        for _ in range(max_new_tokens):
            logits, _ = self.forward(generated)
            next_logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_k > 0:
                v, _ = torch.topk(next_logits, min(top_k, next_logits.shape[-1]))
                next_logits[next_logits < v[:, -1:]] = float("-inf")

            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim=-1)

        return generated

In [9]:
oversize = SpectralLipschitzLinear(8, 8).train()
with torch.no_grad():
    oversize.weight.copy_(4.0 * torch.eye(8))
original_weight = oversize.weight.detach().clone()
features = torch.linspace(-1.0, 1.0, 8).unsqueeze(0)
loss = oversize(oversize(features)).square().mean()
loss.backward()
assert torch.equal(oversize.weight.detach(), original_weight)
assert oversize.weight.grad is not None and torch.isfinite(oversize.weight.grad).all().item()
print({"spectral_reuse_backward": "passed"})

{'spectral_reuse_backward': 'passed'}


### Token task and held-out evaluation

Every token `x` maps to `(x + 3) % 19`. Train and evaluation starts are disjoint. Their trajectories can visit the same states. This tests a deterministic transition learned from token-level signals. ANN's inspected block does not mix tokens across positions, so context-dependent language ability is outside this experiment. Automatic in-forward healing is disabled during the optimizer run. The separate repair experiment runs in evaluation mode.

In [10]:
def trajectories(starts, length=12, alphabet=19):
    return (torch.tensor(starts)[:, None] + 3 * torch.arange(length)[None, :]) % alphabet

train_ids = trajectories(list(range(14)))
test_ids = trajectories(list(range(14, 19)))
assert not set(train_ids[:, 0].tolist()) & set(test_ids[:, 0].tolist())

cfg = ANNConfig.preset_tiny(
    vocab_size=19, d_model=32, n_layers=2, max_seq_len=32,
    enable_self_healing=False, max_ponder_iterations=4,
)
model = AdaptiveNeuralNetwork(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.006, weight_decay=0)

@torch.no_grad()
def evaluate(model, tokens):
    model.eval()
    logits, auxiliary = model(tokens)
    scores = logits[:, :-1].flatten(0, 1)
    labels = tokens[:, 1:].flatten()
    return {
        "cross_entropy": F.cross_entropy(scores, labels).item(),
        "accuracy": (scores.argmax(-1) == labels).float().mean().item(),
        "residuals": auxiliary["layer_residuals"],
    }

before = evaluate(model, test_ids)
model.train()
for step in range(24):
    optimizer.zero_grad(set_to_none=True)
    loss, metrics = model.compute_loss(train_ids)
    assert torch.isfinite(loss).item()
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    assert torch.isfinite(grad_norm).item() and grad_norm.item() > 0
    optimizer.step()
after = evaluate(model, test_ids)
assert after["cross_entropy"] < before["cross_entropy"]
assert all(math.isfinite(v) for layer in after["residuals"] for v in layer)
assert all(1 <= len(layer) <= 4 for layer in after["residuals"])
print({"parameters": sum(p.numel() for p in model.parameters()),
       "before": before, "after": after})

{'parameters': 21312, 'before': {'cross_entropy': 2.946484327316284, 'accuracy': 0.09090909361839294, 'residuals': [[0.9992583990097046, 0.9977505207061768, 0.9924599528312683, 0.9732525944709778], [1.000704288482666, 1.000852346420288, 0.9994911551475525, 0.9903462529182434]]}, 'after': {'cross_entropy': 0.21028713881969452, 'accuracy': 1.0, 'residuals': [[0.9481530785560608, 0.8779857158660889, 0.7395408749580383, 0.5476166009902954], [0.9068579077720642, 0.7752029299736023, 0.5795419812202454, 0.38844385743141174]]}}


### Deliberate damage and measured repair

Zero a reproducible subset of adaptive weights on a copy. The built-in diagnosis uses tracked activation variance, and repair initializes the flagged rows and resets tracker statistics. The number of recovered tracker flags and the held-out loss are separate measurements. A reduced flagged-neuron count does not establish recovery of the learned mapping.

In [11]:
from copy import deepcopy

damaged = deepcopy(model).eval()
control = deepcopy(model).eval()
assert control.diagnose().dead_neurons == 0
torch.manual_seed(SEED + 1)
affected = damaged.inject_trauma(dead_ratio=0.25, shock_noise=0.0)
flagged = damaged.diagnose().dead_neurons
assert affected > 0 and flagged == affected
damaged_eval = evaluate(damaged, test_ids)
repair = damaged.heal()
repaired_eval = evaluate(damaged, test_ids)
assert repair.rejuvenated_neurons == affected
assert damaged.diagnose().dead_neurons == 0
assert math.isfinite(repaired_eval["cross_entropy"])
assert abs(evaluate(control, test_ids)["cross_entropy"] - after["cross_entropy"]) < 1e-7

sample = model.generate(test_ids[:2, :3], max_new_tokens=4, top_k=1)
assert sample.shape == (2, 7)
assert torch.equal(sample[:, :3], test_ids[:2, :3])
print({"injected_damage": affected,
       "flags_before_repair": flagged,
       "flags_after_repair": damaged.diagnose().dead_neurons,
       "baseline_loss": after["cross_entropy"],
       "damaged_loss": damaged_eval["cross_entropy"],
       "repaired_loss": repaired_eval["cross_entropy"],
       "loss_improved_over_damaged": repaired_eval["cross_entropy"] < damaged_eval["cross_entropy"],
       "decoded_tokens": sample.tolist()})

{'injected_damage': 48, 'flags_before_repair': 48, 'flags_after_repair': 0, 'baseline_loss': 0.21028713881969452, 'damaged_loss': 0.5806212425231934, 'repaired_loss': 0.5770143270492554, 'loss_improved_over_damaged': True, 'decoded_tokens': [[14, 17, 1, 4, 7, 10, 13], [15, 18, 2, 5, 8, 11, 14]]}


### Limits

The test covers one seed, a small synthetic task, a bounded latent iteration, and a specific injected fault. A useful extension is to compare matched models with and without adaptive gain, latent iteration, and post-update repair across seeds. Measure predictions on tasks that require actual sequence context before making language modeling claims.